# Aula 02 — Limpeza, Tokenização e Normalização · acompanhamento

Notebook para rodar célula a célula enquanto você assiste à vídeo-aula. Os blocos seguem a ordem dos slides.

**Origem:** `IAST-ProcessamentodeLinguagemNatural-Fase04_aula02-v1.pptx`

**Kernel:** `.venv` do projeto. Tudo que esta aula usa já está instalado, **exceto o slide 40** (ver aviso lá).

## Roteiro

| Bloco | Slides | Assunto |
|---|---|---|
| 1 | 13–20 | Limpeza: regex, HTML, URLs, contrações, acentos |
| 2 | 30–40 | Tokenização: `split()`, NLTK, spaCy, frases, BPE |
| 3 | 50–54 | Normalização: stemming, lemmatization, stop words |
| 4 | 57 | O pipeline final costurado |

## ⚠️ Cinco slides têm a saída esperada errada

Rodei o código dos 15 slides neste ambiente antes de montar o notebook. Os slides **13, 20, 31, 50 e 57** trazem comentários de saída que **não batem** com o que o código produz. Não é erro seu nem do ambiente — em cada um desses pontos há uma célula marcada **"O que você vai ver de verdade"** com o resultado real e o motivo da diferença.

Duas dessas divergências são as partes mais instrutivas da aula: elas expõem falhas de desenho no pipeline de limpeza que o próprio slide não percebe.

> **Correção aplicada:** o slide 14 baixa `punkt`. Desde o NLTK 3.8.2 o `word_tokenize` exige `punkt_tab`. A linha original está preservada, comentada, ao lado da corrigida.

---
# Bloco 1 · Limpeza (slides 13–20)

## Slide 13 — Regex em Python

**O conceito:** a ordem das substituições é o desenho do pipeline. Remover HTML **antes** das URLs importa, porque uma URL dentro de um atributo `href` sumiria junto com a tag. E o filtro de caracteres vem por último, depois que as estruturas já foram extraídas.

**Repare na classe de caracteres do passo 4:** `[^a-záàâãä...ç\s]` lista **apenas minúsculas**. Guarde isso — é a origem do primeiro bug da aula.

In [1]:
import re
# Padrões básicos úteis em pré-processamento de texto
texto = "<p>Adorei!! https://link.com vc tem que ver isso :)</p>"
# Remover tags HTML
limpo = re.sub(r'<[^>]+>', ' ', texto)
# 'Adorei!! https://link.com vc tem que ver isso :)'
# Remover URLs
limpo = re.sub(r'http\S+|www\.\S+', ' ', limpo)
# 'Adorei!!  vc tem que ver isso :)'
# Remover e-mails
limpo = re.sub(r'\S+@\S+', ' ', limpo)
# Manter apenas letras (incluindo acentuadas) e espaços
limpo = re.sub(r'[^a-záàâãäéèêëíìîïóòôõöúùûüç\s]', ' ', limpo)
# 'Adorei   vc tem que ver isso  '
# Colapsar espaços múltiplos
limpo = re.sub(r'\s+', ' ', limpo).strip()
# 'Adorei vc tem que ver isso'
print(repr(limpo))

'dorei vc tem que ver isso'


### ⚠️ O que você vai ver de verdade

| | resultado |
|---|---|
| slide diz | `'Adorei vc tem que ver isso'` |
| **real** | `'dorei vc tem que ver isso'` |

**O `A` maiúsculo é comido.** A classe `[^a-záàâã...]` permite só minúsculas, e neste slide o `lower()` ainda não rodou — então todo caractere maiúsculo cai na negação e vira espaço.

Não é curiosidade: numa base real isso destrói silenciosamente a primeira letra de **todo nome próprio, toda sigla e todo início de frase**. `São Paulo` viraria `ão aulo`.

O slide 20 corrige sem comentar: lá o `lower()` é o **passo 1**, antes do filtro. É por isso que a ordem do pipeline é o conteúdo da aula, não detalhe de implementação.

### Suas anotações

_(o que o professor falou aqui)_

## Slide 14 — Setup do ambiente

**O conceito:** esta célula monta o estado do resto da aula — imports, `nlp` do spaCy e o corpus `reviews`.

Repare na escolha do corpus: são reviews **propositalmente sujas**, com HTML, caixa alta, abreviação de internet (`vc`, `cmprar`, `tbm`), emoji, emoticon (`>:(`) e URL. Cada sujeira dessas existe para ser atacada por um slide específico adiante.

In [2]:
import re
import unicodedata
import nltk
import spacy
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import RSLPStemmer
# Recursos NLTK em português
# nltk.download('punkt', quiet=True)      # linha original do slide — ver nota no topo
nltk.download('punkt_tab', quiet=True)    # exigido pelo word_tokenize desde o NLTK 3.8.2
nltk.download('stopwords', quiet=True)
nltk.download('rslp', quiet=True)
# Modelo spaCy em português
# python -m spacy download pt_core_news_lg
nlp = spacy.load('pt_core_news_lg')
# Corpus exemplo: reviews sujas reais
reviews = [
    "<p>ADOREI o produto!! vc precisa cmprar tbm 😍</p>",
    "Pessimo!! veio quebrado >:( https://reclame.com",
    "Mt bom mesmo, entrega rapida! 5 estrelas pra esse vendedor.",
    "Não gostei :( voltei o item, atendimento foi pessimo tbm.",
]

### Suas anotações

_(o que o professor falou aqui)_

## Slide 15 — Limpeza: HTML, URLs e contrações

**O conceito:** o dicionário `CONTRACTIONS` é conhecimento de domínio codificado à mão. Não existe biblioteca que saiba que `tbm` é `também` em review de e-commerce brasileiro — isso você escreve, e a qualidade da sua lista vira qualidade do resultado.

**Repare no mecanismo:** `texto.split()` quebra por espaço e o `CONTRACTIONS.get(p, p)` exige **igualdade exata** do token. Guarde isso — é a origem do segundo bug da aula, que aparece no slide 20.

In [3]:
# Mapa de contrações comuns em PT-BR informal
CONTRACTIONS = {
    'vc': 'você', 'tb': 'também', 'tbm': 'também',
    'pq': 'por que', 'obg': 'obrigado', 'blz': 'beleza',
    'td': 'tudo', 'msm': 'mesmo', 'vlw': 'valeu',
    'n': 'não', 'naum': 'não', 'cmprar': 'comprar',
    'mt': 'muito', 'pra': 'para'
}
def limpar_texto(texto):
    texto = texto.lower()                                  # 1. Lowercase
    texto = re.sub(r'<[^>]+>', ' ', texto)                 # 2. HTML
    texto = re.sub(r'http\S+|www\.\S+', ' ', texto)        # 3. URLs
    texto = re.sub(r'\S+@\S+', ' ', texto)                 # 4. E-mails
    # 5. Expande contrações
    palavras = texto.split()
    palavras = [CONTRACTIONS.get(p, p) for p in palavras]
    return ' '.join(palavras)
print(limpar_texto(reviews[0]))
# Saída: 'adorei o produto!! você precisa comprar também 😍'

adorei o produto!! você precisa comprar também 😍


### Suas anotações

_(o que o professor falou aqui)_

## Slide 18 — Tratamento de acentos

**O conceito:** `NFKD` decompõe cada caractere acentuado em letra-base + marca de acento separada; o filtro `unicodedata.combining(c)` então descarta as marcas. É normalização Unicode, não substituição de string.

**O porquê da decisão ser "sensível ao downstream"** — o título do slide é a parte importante:

- **Busca / matching:** tirar acento **ajuda**, porque o usuário digita `sao paulo` e quer achar `São Paulo`.
- **NER:** tirar acento **atrapalha**, porque o modelo do spaCy foi treinado em português acentuado. `São Paulo` sem acento pode deixar de ser reconhecido como local.
- **Embeddings / RAG:** mesma lógica do NER — o modelo espera texto natural. Não tire.

Por isso o slide 20 transforma isso num parâmetro opcional em vez de um passo fixo.

In [4]:
import unicodedata
def remove_acentos(texto):
    """Remove acentos: 'maravilhoso' → 'maravilhoso'.
    Útil para busca; perigoso para NER."""
    nfkd = unicodedata.normalize('NFKD', texto)
    return ''.join(c for c in nfkd if not unicodedata.combining(c))
# Exemplos
print(remove_acentos('São Paulo'))      # 'Sao Paulo'
print(remove_acentos('experiência'))    # 'experiencia'
print(remove_acentos('coração'))        # 'coracao'

Sao Paulo
experiencia
coracao


### Suas anotações

_(o que o professor falou aqui)_

## Slide 20 — `limpar_normalizar()`: a função que costura tudo

**O conceito:** é o slide 15 com três melhorias de engenharia, e vale nomear cada uma:

1. **Guarda de tipo** (`if not isinstance(texto, str)`) — devolve string vazia em vez de estourar. Numa coluna de DataFrame com `NaN`, isso é a diferença entre o pipeline rodar e quebrar na linha 40 mil. Esse detalhe vai importar direto no Olist do Tech Challenge.
2. **Lowercase como passo 1** — corrige em silêncio o bug do slide 13.
3. **Acento como parâmetro**, não como passo fixo — a decisão do slide 18 vira configuração.

Repare também que o filtro do passo 6 agora **preserva `.,!?`**. A pontuação sobrevive de propósito, porque o tokenizador do bloco 2 sabe usá-la para separar frases.

In [5]:
def limpar_normalizar(texto, remove_acentos=False):
    """Pipeline modular para texto em PT-BR."""
    if not isinstance(texto, str):
        return ''
    texto = texto.lower()                                       # 1
    texto = re.sub(r'<[^>]+>', ' ', texto)                      # 2
    texto = re.sub(r'http\S+|www\.\S+', ' ', texto)             # 3
    texto = re.sub(r'\S+@\S+', ' ', texto)                      # 4
    palavras = [CONTRACTIONS.get(p, p) for p in texto.split()]  # 5
    texto = ' '.join(palavras)
    texto = re.sub(r'[^a-záàâãäéèêëíìîïóòôõöúùûüç\s.,!?]', ' ', texto)  # 6
    if remove_acentos:                                          # 7
        nfkd = unicodedata.normalize('NFKD', texto)
        texto = ''.join(c for c in nfkd if not unicodedata.combining(c))
    texto = re.sub(r'\s+', ' ', texto).strip()                  # 8
    return texto
# Teste
sujo = '<p>Adorei o produto!! https://exemplo.com VC PRECISA TBM!!!</p>'
print(limpar_normalizar(sujo))
# 'adorei o produto !! você precisa também !!!'

adorei o produto!! você precisa tbm!!!


### ⚠️ O que você vai ver de verdade

| | resultado |
|---|---|
| slide diz | `adorei o produto !! você precisa também !!!` |
| **real** | `adorei o produto!! você precisa tbm!!!` |

Duas diferenças, e a primeira é a lição mais valiosa da aula:

**1. `tbm` não foi expandido.** A expansão do passo 5 usa `texto.split()`, que quebra só por espaço — o token que chega ao dicionário é `tbm!!!`, com a pontuação grudada, e `CONTRACTIONS.get('tbm!!!')` não encontra nada. A contração só é expandida quando está cercada de espaços.

Ou seja: **o passo 5 acontece cedo demais.** Ele depende de tokens limpos, mas roda antes da tokenização de verdade. É um erro de ordenação de pipeline — exatamente o tema do slide. Na review original (`reviews[0]`) funcionou porque lá o `tbm` estava solto entre espaços; bastou colar um `!!!` para o mecanismo falhar em silêncio.

**2. Não há espaço antes de `!!`.** O filtro do passo 6 mantém `!` na lista de permitidos, então nada é substituído por espaço ali. O `produto !!` do comentário não acontece.

**Como você consertaria?** Tokenizar antes de expandir, ou aplicar as contrações com regex de fronteira de palavra (`\b`) em vez de `split()`. Vale anotar sua resposta — é o tipo de decisão que o Tech Challenge pede que você justifique.

### Suas anotações

_(o que o professor falou aqui)_

---
# Bloco 2 · Tokenização (slides 30–40)

## Slide 30 — Família 1: `split()` ingênuo

**O conceito:** o problema do `split()` não é estético. `banco!` e `banco` viram **entradas diferentes no vocabulário** — duas colunas distintas na matriz do slide 56 da aula anterior. O modelo passa a tratar como não relacionadas duas ocorrências da mesma palavra, e o vocabulário incha com variantes que só diferem por pontuação colada.

In [6]:
# split() simples — divide só por espaços
texto = "O Sr. Silva, hoje, tem R$ 1.250,99 no banco!"
tokens = texto.split()
print(tokens)
# ['O', 'Sr.', 'Silva,', 'hoje,', 'tem', 'R$', '1.250,99', 'no', 'banco!']
# Problemas:
# - 'Silva,' (com vírgula) ≠ 'Silva' → tokens DIFERENTES no vocabulário
# - 'banco!' (com !) ≠ 'banco' → tokens DIFERENTES
# - 'hoje,' não separa a vírgula
# Útil só como BASELINE didático.
# Em produção: NUNCA use split() puro.

['O', 'Sr.', 'Silva,', 'hoje,', 'tem', 'R$', '1.250,99', 'no', 'banco!']


### Suas anotações

_(o que o professor falou aqui)_

## Slide 31 — Família 2A: NLTK `word_tokenize`

**O conceito:** o tokenizador não é um `split()` melhorado — ele carrega um modelo estatístico de onde as fronteiras caem, treinado por idioma. É por isso que `language='portuguese'` não é decoração: o modelo sabe que `Sr.` é abreviação e que o ponto ali **não** termina a frase.

A frase de teste foi montada para conter exatamente os casos difíceis: abreviação com ponto, vírgulas coladas, símbolo de moeda, número com separador de milhar e decimal no formato brasileiro.

In [7]:
from nltk.tokenize import word_tokenize, sent_tokenize
texto = "O Sr. Silva, hoje, tem R$ 1.250,99 no banco! Adorei."
# Tokenização de palavras (com language='portuguese')
tokens = word_tokenize(texto, language='portuguese')
print(tokens)
# ['O', 'Sr.', 'Silva', ',', 'hoje', ',', 'tem', 'R$',
#  '1.250,99', 'no', 'banco', '!', 'Adorei', '.']
# ✅ 'Sr.' reconhecido como abreviação
# ✅ Vírgulas separadas como tokens
# ✅ Número 1.250,99 mantido inteiro
# Tokenização de FRASES
frases = sent_tokenize(texto, language='portuguese')
print(frases)
# ['O Sr. Silva, hoje, tem R$ 1.250,99 no banco!', 'Adorei.']

['O', 'Sr.', 'Silva', ',', 'hoje', ',', 'tem', 'R', '$', '1.250,99', 'no', 'banco', '!', 'Adorei', '.']
['O Sr. Silva, hoje, tem R$ 1.250,99 no banco!', 'Adorei.']


### ⚠️ O que você vai ver de verdade

O slide afirma que `R$` é mantido inteiro. **Não é** — sai separado:

| | token de moeda |
|---|---|
| slide diz | `'R$'` |
| **real** | `'R'`, `'$'` |

Todo o resto bate: `Sr.` preservado como abreviação, vírgulas isoladas, `1.250,99` inteiro, e o `sent_tokenize` acerta as duas frases sem se confundir com o ponto de `Sr.`.

O acerto no número e o erro na moeda têm a mesma causa: o tokenizador tem regra para dígitos com separadores, mas trata `$` como pontuação comum. Se símbolo de moeda importar no seu caso, é tratamento manual.

### Suas anotações

_(o que o professor falou aqui)_

## Slide 32 — Família 2B: spaCy

**O conceito:** a diferença para o NLTK não é a qualidade da tokenização — é que o spaCy faz **uma passada só** e devolve o token já anotado com lema, POS e dependência.

No NLTK, cada etapa é uma chamada separada sobre o texto. No spaCy, o `nlp(texto)` executa o pipeline inteiro e o custo é pago uma vez. Quando você for processar dezenas de milhares de reviews, essa diferença deixa de ser estilo e vira tempo de execução.

In [8]:
import spacy
nlp = spacy.load('pt_core_news_lg')
texto = "O Sr. Silva, hoje, tem R$ 1.250,99 no banco!"
# spaCy: tokenização + POS + lema + dependência tudo de uma vez
doc = nlp(texto)
for tok in doc:
    print(f"{tok.text:<12} {tok.lemma_:<12} {tok.pos_}")
# Saída:
# O           o           DET
# Sr.         Sr.         PROPN
# Silva       Silva       PROPN
# ,           ,           PUNCT
# hoje        hoje        ADV
# ...
# 1.250,99    1.250,99    NUM
# no          em o        ADP
# banco       banco       NOUN
# !           !           PUNCT
# 🚀 Vantagem: tokens já vêm com lema, POS, dependência!

O            o            DET
Sr.          Sr.          NOUN
Silva        Silva        PROPN
,            ,            PUNCT
hoje         hoje         ADV
,            ,            PUNCT
tem          ter          VERB
R$           R$           SYM
1.250,99     1.250,99     NUM
no           em o         ADP
banco        banco        NOUN
!            !            PUNCT


### Suas anotações

_(o que o professor falou aqui)_

## Slide 33 — Sentence tokenization

**O conceito:** quebrar em frases é decidir onde um ponto termina uma sentença e onde ele faz parte de uma abreviação, de um número ou de reticências. Repare que `alto...` não vira três frases vazias.

**Por que isso importa para o Tech Challenge:** quebra em frases é a unidade natural de *chunking* para RAG. Quando você decidir como fatiar as reviews do Olist para indexar, é esta operação que está em jogo — e o slide mostra as duas vias, NLTK explícito e spaCy via `.sents`.

In [9]:
# Tokenização de FRASES (não palavras)
from nltk.tokenize import sent_tokenize
texto = ("Adorei o produto. Chegou rápido! "
        "Recomendo a todos. Mas o preço foi alto...")
frases = sent_tokenize(texto, language='portuguese')
for i, f in enumerate(frases, 1):
    print(f'{i}. {f}')
# 1. Adorei o produto.
# 2. Chegou rápido!
# 3. Recomendo a todos.
# 4. Mas o preço foi alto...
# Em spaCy é automático
for sent in nlp(texto).sents:
    print(sent.text)

1. Adorei o produto.
2. Chegou rápido!
3. Recomendo a todos.
4. Mas o preço foi alto...
Adorei o produto.
Chegou rápido!
Recomendo a todos.
Mas o preço foi alto...


### Suas anotações

_(o que o professor falou aqui)_

## Slide 37 — BPE: Byte-Pair Encoding

> Este slide **não tem código executável** — é um bloco de conceito. O conteúdo original está reproduzido abaixo.

O vocabulário começa com caracteres isolados e o algoritmo mescla, iterativamente, o par mais frequente:

```
vocabulário inicial:  'l', 'o', 'w', 'e', 'r', ...
iteração 1:  'l' + 'o'  →  'lo'
iteração 2:  'lo' + 'w' →  'low'
iteração 3:  'e' + 'r'  →  'er'
```

Depois de várias iterações o vocabulário contém *subwords* como `low`, `er`, `est`, `ing`, `##ing`. Resultados:

- `'lowest'` → `['low', 'est']`
- `'newer'` (palavra rara) → `['n', 'ewer']` ou `['new', '##er']`

**Por que isso resolve um problema real:** com vocabulário fixo de palavras inteiras, qualquer palavra não vista no treino vira `<UNK>` e perde todo o significado. Com subwords **não existe OOV** — o pior caso é a palavra ser decomposta em caracteres. É o que permite a um modelo lidar com erro de digitação, neologismo e nome próprio que ele nunca viu.

É também o motivo de o custo de uma chamada de LLM ser medido em *tokens* e não em palavras: são estes os tokens.

### Suas anotações

_(o que o professor falou aqui)_

## Slide 40 — Por que GPT e BERT tokenizam de forma estranha

> ### 🔴 Esta é a única célula da aula que não roda no ambiente atual
>
> Precisa de `transformers` + `torch` (não instalados) e baixa o modelo BERTimbau do HuggingFace na primeira execução. Somando tudo, é da ordem de **2–3 GB**. Para habilitar:
>
> ```powershell
> .venv\Scripts\python -m pip install transformers torch
> ```
>
> O conceito está explicado abaixo, então dá para acompanhar a aula sem rodar.

**O conceito:** é o BPE do slide 37 aplicado a português de verdade. `correríamos` vira `['corre', '##ría', '##mos']` — o `##` marca "continuação da palavra anterior", não início de token novo.

Repare no que isso significa: o modelo aprendeu sozinho, pela estatística do corpus, a separar radical de desinência verbal. Ninguém ensinou gramática a ele. E `inconstitucionalissimamente` estilhaça em nove pedaços porque é rara demais para merecer entrada própria no vocabulário — exatamente o comportamento esperado, não uma falha.

In [ ]:
# Tokenizer do BERTimbau (BERT em PT-BR)
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained('neuralmind/bert-base-portuguese-cased')
print(tok.tokenize('correríamos pela praia'))
# ['corre', '##ría', '##mos', 'pela', 'praia']
print(tok.tokenize('inconstitucionalissimamente'))
# ['in', '##cons', '##ti', '##tu', '##cion', '##ali', '##ss', '##im', '##amente']

### Suas anotações

_(o que o professor falou aqui)_

---
# Bloco 3 · Normalização (slides 50–54)

## Slide 50 — RSLP: o stemmer brasileiro

**O conceito:** stemming corta sufixos por regra. O RSLP (*Removedor de Sufixos da Língua Portuguesa*) é específico para português — aplicar um stemmer de inglês aqui produz lixo.

O resultado **não é uma palavra**: `correndo`, `correu` e `correriam` viram `corr`, que não existe no dicionário. Isso é aceitável quando o objetivo é só agrupar variantes numa mesma coluna do vocabulário, e inaceitável quando alguém vai ler a saída.

In [10]:
from nltk.stem import RSLPStemmer
import nltk
nltk.download('rslp', quiet=True)
stemmer = RSLPStemmer()
exemplos = ['correndo', 'correu', 'correriam',
            'maravilhoso', 'felicidades', 'menino',
            'meninas', 'casinha', 'gatos']
for palavra in exemplos:
    print(f'{palavra:<15} → {stemmer.stem(palavra)}')
# Saída:
# correndo        → corr
# correu          → corr
# correriam       → corr
# maravilhoso     → maravilh
# felicidades     → felic
# menino          → menin
# meninas         → menin
# casinha         → casinh
# gatos           → gat

correndo        → corr
correu          → corr
correriam       → corr
maravilhoso     → maravilh
felicidades     → felic
menino          → menin
meninas         → menin
casinha         → cas
gatos           → gat


### ⚠️ O que você vai ver de verdade

| palavra | slide diz | **real** |
|---|---|---|
| `casinha` | `casinh` | **`cas`** |

Os outros oito batem. O `casinha` é mais interessante que os acertos: o RSLP tem uma regra de **diminutivo** e remove o `-inha` inteiro, chegando ao radical `cas`.

O efeito colateral é que `casinha`, `casa` e `casebre` colapsam no mesmo token. Para agrupar variantes isso é bom; para um modelo que precisasse distinguir "casa" de "casinha" numa descrição de imóvel, é perda de informação irreversível.

### Suas anotações

_(o que o professor falou aqui)_

## Slide 51 — Lemmatization com spaCy

**O conceito:** a comparação direta com o slide 50. Onde o stemmer corta por regra e produz `corr`, o lematizador consulta um modelo linguístico e devolve `correr` — a forma de dicionário, uma palavra real.

O custo é tempo: o stemmer é uma tabela de regras, o lematizador precisa do POS tagging para decidir. Numa base de 100 mil documentos essa diferença é sentida.

**O critério de escolha:** se a saída vai alimentar só uma matriz numérica, stemming basta e é mais rápido. Se alguém vai **ler** os termos — num tópico, numa nuvem de palavras, num insight apresentado ao gestor — precisa ser lematização, porque `corr` não se mostra para ninguém.

In [11]:
import spacy
nlp = spacy.load('pt_core_news_lg')
texto = 'As crianças estavam correndo felizes pela praia maravilhosa.'
doc = nlp(texto)
for token in doc:
    print(f'{token.text:<14} → {token.lemma_}')
# Saída:
# As             → o
# crianças       → criança
# estavam        → estar
# correndo       → correr
# felizes        → feliz
# pela           → por o
# praia          → praia
# maravilhosa    → maravilhoso
# .              → .
# 🎯 Repare: lemmatization PRESERVA palavras válidas
# 'correr' (não 'corr'), 'maravilhoso' (não 'maravilh')

As             → o
crianças       → criança
estavam        → estar
correndo       → correr
felizes        → feliz
pela           → por o
praia          → praia
maravilhosa    → maravilhoso
.              → .


### Suas anotações

_(o que o professor falou aqui)_

## Slide 54 — O que são stop words

**O conceito:** são as palavras de alta frequência e baixo conteúdo semântico — o topo da Lei de Zipf da aula 01. Removê-las encolhe o vocabulário e tira ruído de um modelo de contagem.

**Mas olhe o que está dentro dessa lista de 207.** Rode a célula seguinte à original e confira quais palavras o NLTK considera descartáveis em português.

In [12]:
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords', quiet=True)
STOP_WORDS_PT = set(stopwords.words('portuguese'))
print(f'Total: {len(STOP_WORDS_PT)} stop words')
# Total: 207 stop words
print('Algumas:', list(STOP_WORDS_PT)[:15])
# ['o', 'a', 'os', 'as', 'de', 'do', 'da', 'em', 'no', 'na',
#  'que', 'um', 'uma', 'para', 'com', ...]

Total: 207 stop words
Algumas: ['tive', 'isto', 'muito', 'eram', 'quando', 'elas', 'nossas', 'isso', 'das', 'formos', 'tivessem', 'seu', 'tem', 'também', 'à']


### 🚨 O que a lista de 207 esconde — e por que isso decide o seu Tech Challenge

As palavras de **negação** estão na lista de stop words:

| palavra | está na stop list? |
|---|---|
| `não` | **sim** |
| `nem` | **sim** |
| `sem` | **sim** |
| `mas` | **sim** |
| `nunca` | não |

Consequência verificada neste ambiente:

```
"Não chegou o produto"  →  [chegou, produto]
"Chegou o produto"      →  [chegou, produto]   ← vetores idênticos
```

Numa base de reclamações de e-commerce, remover stop words **inverte o sentido** de uma parte das reviews e o modelo não tem como saber.

**A regra prática que vale para o Tech Challenge:**

- **BoW / TF-IDF** → remover stop words é correto e melhora o resultado.
- **Embeddings densos / RAG** → **não remover**. O modelo de embedding foi treinado em texto natural e usa as palavras funcionais para construir o sentido. Texto mutilado produz vetor mutilado.

Rode a célula abaixo para confirmar você mesmo.

In [13]:
from sklearn.feature_extraction.text import CountVectorizer

for p in ('não', 'nem', 'sem', 'mas', 'nunca'):
    print(f'{p:<8} na stop list? {p in STOP_WORDS_PT}')

vec = CountVectorizer(stop_words=list(STOP_WORDS_PT))
X = vec.fit_transform(['Não chegou o produto', 'Chegou o produto'])
print('\nvocabulário:', vec.get_feature_names_out())
print('negativa   :', X.toarray()[0])
print('positiva   :', X.toarray()[1])
print('idênticos? ', (X.toarray()[0] == X.toarray()[1]).all())

não      na stop list? True
nem      na stop list? True
sem      na stop list? True
mas      na stop list? True
nunca    na stop list? False

vocabulário: ['chegou' 'produto']
negativa   : [1 1]
positiva   : [1 1]
idênticos?  True


### Suas anotações

_(o que o professor falou aqui)_

---
# Bloco 4 · O pipeline final (slide 57)

## Slide 57 — Tudo costurado em uma função

**O conceito:** a aula inteira condensada em quatro passos — limpar, tokenizar, filtrar, normalizar. Repare que o filtro do passo 3 faz **duas** coisas: descarta stop words **e** descarta tokens com 2 caracteres ou menos, o que varre de uma vez a pontuação isolada que o tokenizador produziu.

⚠️ Esta célula **redefine `reviews`** — o slide 14 usou o mesmo nome para as quatro reviews sujas. Se voltar ao Bloco 1 depois daqui, rode o slide 14 de novo antes de seguir.

⚠️ Depende de `limpar_normalizar` (slide 20) e de `CONTRACTIONS` (slide 15). Se abriu o notebook agora, rode os blocos anteriores primeiro.

In [14]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import RSLPStemmer
STOP_WORDS_PT = set(stopwords.words('portuguese'))
stemmer = RSLPStemmer()
def pipeline_nlp(textos: list[str]) -> list[list[str]]:
    """Lista de textos PT → lista de listas de tokens limpos."""
    resultado = []
    for texto in textos:
        texto = limpar_normalizar(texto)                              # 1
        tokens = word_tokenize(texto, language='portuguese')          # 2
        tokens = [t for t in tokens                                   # 3
                  if t not in STOP_WORDS_PT and len(t) > 2]
        tokens = [stemmer.stem(t) for t in tokens]                    # 4
        resultado.append(tokens)
    return resultado
reviews = ['Adorei o produto, entrega rapidíssima!',
           'Não gostei, veio quebrado e atrasado.']
print(pipeline_nlp(reviews))
# [['ador', 'produt', 'entreg', 'rap'],
#  ['gost', 'vei', 'quebr', 'atras']]

[['ador', 'produt', 'entreg', 'rapid'], ['gost', 'vei', 'quebr', 'atras']]


### ⚠️ O que você vai ver de verdade

| | resultado |
|---|---|
| slide diz | `[['ador','produt','entreg',`**`'rap'`**`], ['gost','vei','quebr','atras']]` |
| **real** | `[['ador','produt','entreg',`**`'rapid'`**`], ['gost','vei','quebr','atras']]` |

`rapidíssima` vira `rapid`, não `rap` — o RSLP remove o superlativo `-íssima` e para no radical válido.

**Mas olhe a segunda lista, que é o ponto que importa.** A entrada era *"Não gostei, veio quebrado e atrasado"* e a saída é `['gost', 'vei', 'quebr', 'atras']`. **O `não` sumiu.** Sem ele, `gost` é indistinguível de uma review que dissesse "gostei".

Este é o pipeline correto da aula, feito exatamente como ensinado — e ele destrói a negação. Não é erro do professor: é a consequência esperada de remover stop words, e o motivo pelo qual esse pipeline **não pode** ser reaproveitado para a base de embeddings do Tech Challenge.

### Suas anotações

_(o que o professor falou aqui)_

---
## O que esta aula deixa em aberto

| Ponto | Onde continua |
|---|---|
| Tokens limpos ainda não são números | BoW e TF-IDF — aula 3 |
| Stemming/lematização colapsam variantes mas não capturam sentido | embeddings — aula 4 |
| BPE só foi visto por cima (slide 37) | volta nos transformers — IA Generativa |
| Contrações expandidas por `split()` falham com pontuação colada | seu exercício; nenhum slide corrige |

## Conexão com o Tech Challenge

Esta aula é, literalmente, o **Requisito 1** do desafio — *"limpeza e preparação dos textos"* e *"tratamento de valores ausentes"*. Três coisas daqui vão direto para o projeto:

1. **A guarda de tipo do slide 20** (`if not isinstance(texto, str)`) é o que faz o pipeline sobreviver à coluna `review_comment_message` do Olist, onde a maioria dos registros é nula.
2. **A quebra em frases do slide 33** é a base da sua estratégia de *chunking* para indexação.
3. **A decisão de acento do slide 18** se repete para o embedding: texto natural, sem remoção.

E o alerta que atravessa a aula inteira: **o `pipeline_nlp` do slide 57 é o pipeline certo para a aula 3 e o pipeline errado para o RAG.** Stemming e remoção de stop words servem a modelos de contagem. Para retrieval denso, o texto vai para o embedding como o cliente escreveu — sujeira de HTML e URL removidas, sim; negação e palavras funcionais, jamais.

Vale montar **dois caminhos de preparação** no Tech Challenge, partindo do mesmo texto cru, e justificar a existência dos dois no README. Essa justificativa é item de nota.